# Refinamiento Music Brainz

In [1]:
from pyspark.sql import SparkSession, functions as F, types as T, Window

spark = (SparkSession.builder.appName("refinamiento_musicbrainz")
         .config("spark.driver.memory","2g").enableHiveSupport().getOrCreate())

spark.conf.set("spark.sql.legacy.timeParserPolicy", "LEGACY")
spark.conf.set("spark.sql.parquet.datetimeRebaseModeInWrite", "LEGACY")

RAW = "/Obligatorio/landing/musicbrainz"
REFINED = "/Obligatorio/refined"
def I(): return T.IntegerType()
def S(): return T.StringType()
def f(n,t): return T.StructField(n,t,True)
schemas = {
 "area":[f("id",I()),f("mbid",S()),f("name",S())],
 "artist":[f("id",I()),f("mbid",S()),f("name",S()),f("sort_name",S()),f("begin_year",I()),f("end_year",I()),f("type",I()),f("area",I())],
 "artist_type":[f("id",I()),f("name",S())],
 "country_area":[f("area_id",I())],
 "event":[f("id",I()),f("mbid",S()),f("name",S()),f("begin_year",I()),f("begin_month",I()),f("begin_day",I()),f("end_year",I()),f("end_month",I()),f("end_day",I()),f("type",I()),f("cancelled",S())],
 "event_type":[f("id",I()),f("name",S())],
 "l_area_area":[f("id",I()),f("parent_area_id",I()),f("child_area_id",I())],
 "l_artist_event":[f("id",I()),f("artist_id",I()),f("event_id",I())],
 "l_event_place":[f("id",I()),f("event_id",I()),f("place_id",I())],
 "place":[f("id",I()),f("mbid",S()),f("name",S()),f("type",I()),f("area",I()),f("coordinates",S())],
}
mb = {n: spark.read.option("header",True).schema(T.StructType(fl)).csv(f"{RAW}/mb_{n}.csv")
      for n,fl in schemas.items()}

def limpiar(c):
    s = F.trim(F.col(c).cast("string"))
    return F.when(s.isin(["","\\N"]), None).otherwise(s)
def norm(c): return F.lower(F.trim(F.regexp_replace(F.col(c), r"\s+", " ")))
def country_key_str(colname): return F.sha2(F.lower(F.trim(F.regexp_replace(F.col(colname), r"\s+", " "))), 256)
print("Tablas:", list(mb.keys()))



Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


2026-06-24T00:25:20,601 WARN [Thread-4] org.apache.hadoop.util.NativeCodeLoader - Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Tablas: ['area', 'artist', 'artist_type', 'country_area', 'event', 'event_type', 'l_area_area', 'l_artist_event', 'l_event_place', 'place']


In [2]:
# Propagamos el país por la jerarquía de áreas (igual que en el exploratorio).
laa = mb["l_area_area"].select("parent_area_id","child_area_id")
area_country = mb["country_area"].select("area_id", F.col("area_id").alias("country_area_id"))
for i in range(10):
    nuevos = (laa.join(area_country.withColumnRenamed("area_id","parent_area_id"),"parent_area_id")
                 .select(F.col("child_area_id").alias("area_id"),"country_area_id"))
    antes = area_country.count()
    area_country = area_country.unionByName(nuevos).dropDuplicates(["area_id"])
    if area_country.count() == antes: break

# Nombre del país por área-país, y clave determinística (inglés -> coincide con OpenFlights)
area_country = (area_country
    .join(mb["area"].select(F.col("id").alias("country_area_id"), F.col("name").alias("country_name")),
          "country_area_id","left")
    .withColumn("country_id", country_key_str("country_name"))
    .select("area_id","country_id","country_name"))
print("Áreas resolubles a país:", area_country.count())

artist_types = mb["artist_type"].select(F.col("id").alias("type"), F.col("name").alias("artist_type"))
event_types  = mb["event_type"].select(F.col("id").alias("type"), F.col("name").alias("event_type"))
area_name    = mb["area"].select(F.col("id").alias("area"), F.col("name").alias("area_name"))



[Stage 780:>                                                        (0 + 2) / 2]

Áreas resolubles a país: 119893


In [3]:
# city_id determinístico = sha2(pais|ciudad), igual fórmula que OpenFlights -> claves consistentes.
place_refined = (
    mb["place"]
    .withColumn("name", limpiar("name"))
    .withColumn("latitude",  F.regexp_extract("coordinates", r"\(([-0-9.]+),([-0-9.]+)\)",1).cast("double"))
    .withColumn("longitude", F.regexp_extract("coordinates", r"\(([-0-9.]+),([-0-9.]+)\)",2).cast("double"))
    # anular las 15 coordenadas fuera de rango
    .withColumn("latitude",  F.when(F.col("latitude").between(-90,90) & F.col("longitude").between(-180,180), F.col("latitude")))
    .withColumn("longitude", F.when(F.col("latitude").isNotNull(), F.col("longitude")))
    .join(area_name, "area", "left")                 # nombre de la ciudad/área del lugar
    .join(area_country, F.col("area")==area_country.area_id, "left")  # país resuelto
    .withColumn("city_name", F.col("area_name"))
    .withColumn("city_id",
        F.when(F.col("country_name").isNotNull() & F.col("city_name").isNotNull(),
               F.sha2(F.concat_ws("|", norm("country_name"), norm("city_name")), 256)))
    .select(F.col("id").alias("place_id"), "city_id", "city_name", "country_id", "country_name", "latitude","longitude")
)
print("place refinados:", place_refined.count())



[Stage 853:>                                                        (0 + 2) / 2]

place refinados: 80651


In [4]:
artists_clean = (
    mb["artist"]
    .withColumn("name", limpiar("name")).withColumn("mbid", limpiar("mbid"))
    .filter(F.col("id").isNotNull() & F.col("mbid").isNotNull() & F.col("name").isNotNull())
    .dropDuplicates(["id"])
)
dim_artist = (
    artists_clean
    .join(artist_types, "type", "left")
    .join(area_country.select(F.col("area_id").alias("area"),
                              F.col("country_id").alias("origin_country_id")), "area", "left")
    .select(
        F.col("id").alias("artist_id"),
        "mbid",
        F.col("name").alias("artist_name"),
        "artist_type",
        "origin_country_id",
        "begin_year",
        "end_year",
        (F.col("end_year").isNull()).alias("is_active"),
    )
)
print("dim_artist:", dim_artist.count())
dim_artist.write.mode("overwrite").parquet(f"{REFINED}/dim_artist")



[Stage 928:>                                                        (0 + 2) / 2]

dim_artist: 567202


In [5]:
# Un lugar representativo por evento (el primero), y de ahí ciudad/país.
ev_place = (
    mb["l_event_place"].dropDuplicates(["event_id"])
    .join(place_refined, "place_id", "left")
    .select("event_id","city_id","country_id")
)
events_clean = (
    mb["event"]
    .withColumn("name", limpiar("name")).withColumn("mbid", limpiar("mbid")).withColumn("cancelled", limpiar("cancelled"))
    .filter(F.col("id").isNotNull() & F.col("mbid").isNotNull() & F.col("name").isNotNull())
    .dropDuplicates(["id"])
)
def fecha(y,m,d):
    full = F.col(y).isNotNull() & F.col(m).isNotNull() & F.col(d).isNotNull()
    return F.when(full, F.to_date(F.concat_ws("-", F.col(y), F.lpad(F.col(m),2,"0"), F.lpad(F.col(d),2,"0"))))

dim_event = (
    events_clean
    .join(event_types, "type", "left")
    .join(ev_place, F.col("id")==ev_place.event_id, "left")
    .withColumn("begin_date", fecha("begin_year","begin_month","begin_day"))
    .withColumn("end_date",   fecha("end_year","end_month","end_day"))
    .select(
        F.col("id").alias("event_id"),
        "mbid",
        F.col("name").alias("event_name"),
        "event_type",
        "city_id",
        "country_id",
        "begin_date",
        "end_date",
        (F.col("cancelled")=="t").alias("cancelled"),
    )
)
print("dim_event:", dim_event.count())
dim_event.write.mode("overwrite").parquet(f"{REFINED}/dim_event")



dim_event: 117931


In [6]:
artist_ids = dim_artist.select("artist_id")
event_ids  = dim_event.select("event_id")
bridge_artist_event = (
    mb["l_artist_event"]
    .filter(F.col("artist_id").isNotNull() & F.col("event_id").isNotNull())
    .dropDuplicates(["artist_id","event_id"])          # quita los 5.888 pares repetidos
    .join(artist_ids, "artist_id", "left_semi")        # integridad referencial
    .join(event_ids,  "event_id",  "left_semi")
    .select("artist_id","event_id")
)
print("bridge_artist_event:", bridge_artist_event.count())
bridge_artist_event.write.mode("overwrite").parquet(f"{REFINED}/bridge_artist_event")

bridge_artist_event: 234845


In [7]:
# Enriquecemos cada par artista-evento con país de origen del artista y país/ciudad del evento.
ae_full = (
    bridge_artist_event
    .join(dim_artist.select("artist_id","origin_country_id"), "artist_id", "left")
    .join(dim_event.select("event_id",
                           F.col("country_id").alias("event_country_id"),
                           F.col("city_id").alias("event_city_id"),
                           "begin_date"), "event_id", "left")
    .withColumn("event_date_id", F.date_format("begin_date","yyyyMMdd").cast("int"))
)

fact_artist_event = ae_full.select(
    "artist_id","event_id","event_date_id",
    F.col("origin_country_id").alias("artist_origin_country_id"),
    "event_country_id","event_city_id")
print("fact_artist_event:", fact_artist_event.count())
fact_artist_event.write.mode("overwrite").parquet(f"{REFINED}/fact_artist_event")

# Flujo musical entre países: país origen del artista -> país del evento, por año.
fact_country_music_flow = (
    ae_full
    .withColumn("year", F.year("begin_date"))
    .filter(F.col("origin_country_id").isNotNull() & F.col("event_country_id").isNotNull())
    .groupBy(F.col("origin_country_id").alias("artist_origin_country_id"),
             "event_country_id","year")
    .agg(F.countDistinct("artist_id").alias("artists_count"),
         F.countDistinct("event_id").alias("events_count"))
    .select("artist_origin_country_id","event_country_id","year","artists_count","events_count")
)
print("fact_country_music_flow:", fact_country_music_flow.count())
fact_country_music_flow.write.mode("overwrite").parquet(f"{REFINED}/fact_country_music_flow")

fact_artist_event: 234845


fact_country_music_flow: 13220


In [8]:
# Países de MusicBrainz (en inglés -> mismas claves que OpenFlights). Se unen a dim_country.
paises_mb = (area_country.select("country_id","country_name").dropDuplicates(["country_id"])
    .withColumn("country_iso", F.lit(None).cast("string"))
    .withColumn("source", F.lit("musicbrainz"))
    .select("country_id","country_name","country_iso","source"))
prev = spark.read.parquet(f"{REFINED}/dim_country")
dim_country = prev.unionByName(paises_mb).dropDuplicates(["country_id"])
dim_country.write.mode("overwrite").parquet(f"{REFINED}/dim_country_tmp")
spark.read.parquet(f"{REFINED}/dim_country_tmp").write.mode("overwrite").parquet(f"{REFINED}/dim_country")
print("dim_country:", spark.read.parquet(f"{REFINED}/dim_country").count())

# dim_city (1ª versión, desde los lugares de MusicBrainz). Lat/lon se enriquecen luego.
dim_city = (
    place_refined
    .filter(F.col("city_id").isNotNull())
    .select("city_id","city_name","country_id")
    .dropDuplicates(["city_id"])
    .withColumn("latitude",  F.lit(None).cast("double"))
    .withColumn("longitude", F.lit(None).cast("double"))
    .select("city_id","city_name","country_id","latitude","longitude")
)
print("dim_city:", dim_city.count())
dim_city.write.mode("overwrite").parquet(f"{REFINED}/dim_city")

dim_country: 397


[Stage 2146:>                                                       (0 + 2) / 2]

dim_city: 11325


In [9]:
for t in ["dim_artist","dim_event","bridge_artist_event","fact_artist_event",
          "fact_country_music_flow","dim_country","dim_city"]:
    df = spark.read.parquet(f"{REFINED}/{t}")
    print(f"{t:26} filas={df.count():>8}  columnas={len(df.columns)}")

print("\nEventos por país (top):")
(spark.read.parquet(f"{REFINED}/dim_event")
    .join(spark.read.parquet(f"{REFINED}/dim_country"),"country_id","left")
    .groupBy("country_name").count().orderBy(F.desc("count")).show(10, truncate=False))



dim_artist                 filas=  567202  columnas=8
dim_event                  filas=  117931  columnas=9
bridge_artist_event        filas=  234845  columnas=2
fact_artist_event          filas=  234845  columnas=6
fact_country_music_flow    filas=   13220  columnas=5
dim_country                filas=     397  columnas=4
dim_city                   filas=   11325  columnas=5

Eventos por país (top):
+--------------+-----+
|country_name  |count|
+--------------+-----+
|United States |38586|
|null          |15852|
|United Kingdom|11470|
|Germany       |10470|
|Japan         |6405 |
|Canada        |5215 |
|Netherlands   |4172 |
|France        |3893 |
|Belgium       |2549 |
|Israel        |1518 |
+--------------+-----+
only showing top 10 rows

